# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
#loading the data from last week
!pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

base = "hf://datasets/FlyRank/internship-warehouse"

df_content = con.sql(f"SELECT * FROM read_parquet('{base}/dim_content.parquet')").df()
df_clients = con.sql(f"SELECT * FROM read_parquet('{base}/dim_clients.parquet')").df()
df_march = con.sql(f"SELECT * FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/data_0.parquet')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
agg_df = df_march.groupby(['client_hash_id', 'content_hash_id']).agg(
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    gsc_sum_position=('gsc_sum_position', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
).reset_index()

agg_df['gsc_avg_position'] = agg_df['gsc_sum_position'] / agg_df['gsc_impressions']

print(f"agg_df shape: {agg_df.shape}")
agg_df.head()

agg_df shape: (331437, 8)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,ga4_sessions,ga4_engaged_sessions,gsc_avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,0,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,0,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,0,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0,0,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,0,0,0,NaN


## 1. My rule and its reason codes

**Signal 1: CTR vs. Position — CONFIRMED**

Checked whether pages ranking better on Google (lower average position) get a higher click-through-rate, as FlyRank's CTR-fix flag assumes.

| Position bucket | n (pages) | Avg CTR |
|---|---|---|
| 1-3 (top) | 18,860 | 1.17% |
| 4-10 | 83,288 | 0.49% |
| 11-20 | 29,922 | 0.33% |
| 21+ | 44,668 | 0.20% |

CTR drops consistently and monotonically as position worsens — confirming the assumption behind FlyRank's CTR-fix logic holds true in this data.

In [3]:
#Pages that actually got impressions
signal_df = agg_df[agg_df['gsc_impressions'] > 0].copy()

#CTR for each page
signal_df['ctr'] = signal_df['gsc_clicks'] / signal_df['gsc_impressions']

# Bucket pages by their average position
def position_bucket(pos):
    if pos <= 3:
        return '1-3 (top)'
    elif pos <= 10:
        return '4-10'
    elif pos <= 20:
        return '11-20'
    else:
        return '21+'

signal_df['position_bucket'] = signal_df['gsc_avg_position'].apply(position_bucket)

# Build the bucket table: average CTR per position bucket, with row counts (n)
ctr_position_table = signal_df.groupby('position_bucket').agg(
    n=('ctr', 'count'),
    avg_ctr=('ctr', 'mean')
).reindex(['1-3 (top)', '4-10', '11-20', '21+'])

print(ctr_position_table)

                     n   avg_ctr
position_bucket                 
1-3 (top)        18860  0.011696
4-10             83288  0.004873
11-20            29922  0.003285
21+              44668  0.001952


In [7]:
#signal 2: Engagement rate
#filtering to seessions with actual engagement
engagement_df = agg_df[agg_df['ga4_engaged_sessions'] > 0].copy()

engagement_df['engagement_rate'] = engagement_df['ga4_engaged_sessions'] / engagement_df['ga4_sessions'] # rate = engaged sessions / total sessions

def engagement_bucket(rate):
    if rate >= 0.6:
        return 'high (60%+)'
    elif rate >= 0.3:
        return 'medium (30-59%)'
    elif rate >= 0.1:
        return 'low (10-29%)'
    else:
        return 'very low (<10%)'

engagement_df['engagement_bucket'] = engagement_df['engagement_rate'].apply(engagement_bucket)

engagement_table = engagement_df.groupby('engagement_bucket').agg(
    n=('engagement_rate', 'count'),
    avg_sessions=('ga4_sessions', 'mean')
).reindex(['high (60%+)', 'medium (30-59%)', 'low (10-29%)', 'very low (<10%)'])

print(engagement_table)



                      n  avg_sessions
engagement_bucket                    
high (60%+)         784       1.21301
medium (30-59%)    1520      3.098026
low (10-29%)       3650     15.327123
very low (<10%)    7857     81.597047


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.